# Prior correction via elastic net

In this prior inflation, we solve an optimization problem: we want to find the correction factor for each of the PCA mode such that the binary observation of thawed and frozen is in distribution to the "synthetic observations". This is done with the following assumptions: we assume that the likelihood function is a multivariate normal distribution, and we assume that the correction factor should be heavily regularized by either Lasso or Ridge criterion

In [ ]:
# get the synthetic binary observation
import torch
from src.priorCorrection import PriorCorrection
from src.observationOperator import latent_temperature_operator_enthalpy, temperature_binary_soft_operator

Eb_mean_data = torch.from_numpy(md.X_mean)
Eb_std_data = torch.from_numpy(md.X_std)
Tpmp = torch.from_numpy(md.pmp).flatten()
Eb_V = torch.from_numpy(md.pca_x.components_)
Eb_star = torch.from_numpy(md.X_reduced.T)
combined_mask = torch.from_numpy(md.combined_mask)

dtype = torch.float32

Eb_mean_data = torch.as_tensor(
    md.X_mean,
    dtype=dtype,
)

Eb_std_data = torch.as_tensor(
    md.X_std,
    dtype=dtype,
)

Tpmp = torch.as_tensor(
    md.pmp,
    dtype=dtype,
).flatten()

Eb_V = torch.as_tensor(
    md.pca_x.components_,
    dtype=dtype,
)

Eb_star = torch.as_tensor(
    md.X_reduced.T,
    dtype=dtype,
)

delta_T = latent_temperature_operator_enthalpy(Eb_V, Eb_star, Eb_mean_data, Eb_std_data, Tpmp, 
                                   method="relaxation", 
                                   epsilon=md.X_epsilon)
binary_class = temperature_binary_soft_operator(delta_T, beta=1)

pc = PriorCorrection()
pc.load_simulation_var(Eb_V, Eb_star, md.pca_x.explained_variance_, 
                       Eb_mean_data, Eb_std_data, 
                       Tpmp, 
                       method="relaxation", epsilon=md.X_epsilon)

# plot the actual basal thermal state observations
obs_base = np.nan * np.ones((md.nx * md.ny))
obs_base = np.where(md.thawed_mask, 1, np.where(md.frozen_mask, 0, np.nan))
plt.imshow(obs_base.reshape(md.nx, md.ny))
plt.gca().invert_yaxis()
plt.colorbar()

obs_base = torch.tensor(obs_base)
obs_base = torch.as_tensor(
    obs_base,
    dtype=dtype,
)

pc.load_observation(obs_base)

In [ ]:
rho_e = 1000
rho_w = 10000
sigma_sq = 0.0001 # increases this decreases the mahalanobis val
results = pc.solve_MAP(rho_e=rho_e, rho_w=rho_w, sigma_sq=sigma_sq,
                       lr = 0.5, max_eval_per_iter=20, max_iter=100, l1_ratio=0.9,
                       active_modes=range(1,100))

# sensible parameter combo: 10, 10, 0.1; lr = 0.5, l1_ratio=0.9
# sensible parameter combo2: 100, 10, 0.1; lr = 0.5, l1_ratio=0.9
# sensible parameter combo3: 100, 10, 0.001; lr = 0.5, l1_ratio=0.9
# sensible parameter combo4: 10, 10000, 0.0001; lr = 0.5, l1_ratio=0.9
# sensible parameter combo5: 1000, 10000, 0.0001; lr = 0.5, l1_ratio=0.9

# print some results
# "mahalanobis", "regularization_penalty_e", "regularization_penalty_w"
print("mahalanobis:", results["mahalanobis"])
print("regularization_penalty_e:", results["regularization_penalty_e"])
print("regularization_penalty_w:", results["regularization_penalty_w"])

plt.figure(figsize=(15, 4))
plt.subplot(1,3,1)
plt.plot(results['loss_history'])
plt.title('Loss Evolution')

plt.subplot(1,3,2)
plt.plot(results['alpha'])
plt.title('Alpha across the modes')

plt.subplot(1,3,3)
plt.plot(results['raw_coefficient_shift'])
plt.title('raw shift across the modes')


plt.show()

In [ ]:
from itertools import product
from pathlib import Path
from tqdm import tqdm
import torch

rho_e_grid = [10, 30, 100, 300, 1000, 3000]
rho_w_grid = [10, 100, 1000, 3000, 10000, 30000]
sigma_sq_grid = [0.1, 0.03, 0.01, 0.003, 0.001, 0.0003, 0.0001]

parameter_grid = list(product(
    rho_e_grid,
    rho_w_grid,
    sigma_sq_grid,
))

output_path = Path(
    "../data/prior-correction/map_grid_search_results.pt"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

results = []

for rho_e, rho_w, sigma_sq in tqdm(parameter_grid):
    parameters = {
        "rho_e": rho_e,
        "rho_w": rho_w,
        "sigma_sq": sigma_sq,
    }

    try:
        result = pc.solve_MAP(
            rho_e=rho_e,
            rho_w=rho_w,
            sigma_sq=sigma_sq,
            lr=0.5,
            max_eval_per_iter=20,
            max_iter=100,
            l1_ratio=0.9,
            active_modes=range(1, 100),
            show_progress=False,
        )

        finite = (
            torch.isfinite(result["loss"]).all()
            and torch.isfinite(result["alpha"]).all()
        )

        converged = (
            result["convergence_reason"]
            != "maximum iterations reached"
        )

        if not finite:
            status = "non-finite"
        elif not converged:
            status = "not-converged"
        else:
            status = "success"

        results.append({
            **parameters,
            "status": status,
            "result": result,
        })

    except Exception as error:
        results.append({
            **parameters,
            "status": "error",
            "error": repr(error),
            "result": None,
        })

    # Save a checkpoint after every run.
    torch.save(results, output_path)


# Prior inflation via global correction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.ice import enthalpy_to_temperature
from tqdm.auto import tqdm
from gmr import GMM

# ── Configuration ─────────────────────────────────────────────────────────────
N_SAMPLES_BISECT = 500
N_SAMPLES_FINAL  = 2000
P_THAWED         = 95
P_FROZEN         = 5
ERR_EPS          = 1.0
BISECT_TOL       = 1e-3
BISECT_MAX_ITER  = 30
S_MAX            = 10.0
CONSISTENCY_TARGET = 0.90   # accept 90% rather than demanding 100% 
                             # (last 10% is likely mean-biased, unfixable by variance)


# ── Build perturbed GMM ───────────────────────────────────────────────────────
def build_perturbed_gmm(gmm_original, scale_vector):
    new_covs = gmm_original.covariances.copy()
    diag_idx = np.arange(len(scale_vector))
    new_covs[:, diag_idx, diag_idx] *= scale_vector ** 2
    return GMM(
        n_components = gmm_original.n_components,
        priors       = gmm_original.priors.copy(),
        means        = gmm_original.means.copy(),
        covariances  = new_covs,
        random_state = gmm_original.random_state
    )


# ── Batched Tb percentile computation ────────────────────────────────────────
def sample_tb_percentiles(gmm, md, n_samples=N_SAMPLES_FINAL, desc="Sampling"):
    samples  = gmm.sample(n_samples)
    Eb_batch = md.pca_x.inverse_transform(samples)     # vectorized PCA step
    domain_flat = md.domain_mask.flatten()
    Tb_all   = np.zeros((n_samples, int(domain_flat.sum())))

    for i in tqdm(range(n_samples), desc=desc, leave=False):
        Eb_ori = reverse_standardize(Eb_batch[i], md.X_mean, md.X_std,
                                     method='relaxation', epsilon=md.X_epsilon)
        Tb_ori = enthalpy_to_temperature(Eb_ori, md.pmp, istorch=False)
        Tb_all[i] = Tb_ori[domain_flat]

    return (np.nanpercentile(Tb_all, P_FROZEN,  axis=0),
            np.nanpercentile(Tb_all, P_THAWED, axis=0))


# ── Consistency scores ────────────────────────────────────────────────────────
def consistency_scores(Tb_plow, Tb_phigh, md, err_eps=ERR_EPS):
    flat_pmp   = md.pmp.flatten()
    full_shape = flat_pmp.shape[0]
    domain_idx = np.where(md.domain_mask.flatten())[0]

    Tb_phigh_full = np.full(full_shape, np.nan)
    Tb_plow_full  = np.full(full_shape, np.nan)
    Tb_phigh_full[domain_idx] = Tb_phigh
    Tb_plow_full[domain_idx]  = Tb_plow

    thawed_m = md.thawed_mask.flatten().astype(bool)
    frozen_m = md.frozen_mask.flatten().astype(bool)

    thawed_ok = (Tb_phigh_full[thawed_m] - flat_pmp[thawed_m]) >= -err_eps
    frozen_ok = (Tb_plow_full[frozen_m]  - flat_pmp[frozen_m]) <= -err_eps

    frac_thawed = np.mean(thawed_ok) if thawed_m.any() else 1.0
    frac_frozen = np.mean(frozen_ok) if frozen_m.any() else 1.0
    return frac_thawed, frac_frozen


# ── Phase 1: global bisection (all dims scaled equally) ──────────────────────
def find_global_scale(md, target=CONSISTENCY_TARGET, verbose=True):
    """
    Bisect a single global scalar s applied to ALL PCA dims simultaneously.
    This is a 1D bisection — cheap and handles the joint coverage problem.
    Returns s_global.
    """
    print("── Phase 1: Global scale bisection ─────────────────────────────────")

    def evaluate(s_global):
        sv    = np.full(md.pca_x.n_components_, s_global)
        gmm_p = build_perturbed_gmm(md.gmm_prop, sv)
        Tb_pl, Tb_ph = sample_tb_percentiles(gmm_p, md,
                                              n_samples=N_SAMPLES_BISECT,
                                              desc=f"  s={s_global:.4f}")
        ft, ff = consistency_scores(Tb_pl, Tb_ph, md)
        return ft, ff

    # Quick check: is S_MAX even sufficient?
    ft_max, ff_max = evaluate(S_MAX)
    print(f"  s={S_MAX:.1f} → Thawed: {ft_max:.3f}  Frozen: {ff_max:.3f}")
    if ft_max < target:
        print(f"  ⚠ WARNING: even S_MAX={S_MAX} cannot reach target={target:.2f}.")
        print(f"    The remaining {1-ft_max:.1%} thawed deficit is likely mean-biased.")
        print(f"    Returning S_MAX as best achievable global scale.")
        return S_MAX, ft_max, ff_max

    # Bisect
    s_lo, s_hi = 1.0, S_MAX
    s_best     = S_MAX
    ft_best, ff_best = ft_max, ff_max

    bisect_bar = tqdm(range(BISECT_MAX_ITER), desc="Global bisect", unit="iter")
    for _ in bisect_bar:
        s_mid    = 0.5 * (s_lo + s_hi)
        ft, ff   = evaluate(s_mid)
        thawed_ok = ft >= target
        frozen_ok = ff >= target

        bisect_bar.set_postfix({
            "s":      f"{s_mid:.4f}",
            "thawed": f"{ft:.3f}",
            "frozen": f"{ff:.3f}",
            "ok":     "✓" if (thawed_ok and frozen_ok) else "✗"
        })

        if thawed_ok and frozen_ok:
            s_best, ft_best, ff_best = s_mid, ft, ff
            s_hi = s_mid
        else:
            s_lo = s_mid

        if (s_hi - s_lo) < BISECT_TOL:
            break

    print(f"\n  Global scale found: s* = {s_best:.4f}  "
          f"(Thawed: {ft_best:.3f}  Frozen: {ff_best:.3f})")
    return s_best, ft_best, ff_best


# ── Phase 2: per-dim refinement from global scale ────────────────────────────
def refine_per_dim(md, s_global, verbose=True):
    """
    Starting from s_global on all dims, try to reduce each dim's scale
    back toward 1.0 (LASSO minimization) while maintaining consistency.
    This is now tractable because we start from a feasible point.
    """
    print("\n── Phase 2: Per-dim refinement (LASSO minimization) ─────────────────")
    n_dims       = md.pca_x.n_components_
    scale_vector = np.full(n_dims, s_global)

    dim_bar = tqdm(range(n_dims), desc="Refining dims", unit="dim")
    for k in dim_bar:

        # Try reducing dim k toward 1.0 via bisection
        s_lo, s_hi = 1.0, scale_vector[k]
        s_best     = scale_vector[k]   # fallback: keep global scale

        for _ in range(BISECT_MAX_ITER):
            s_mid = 0.5 * (s_lo + s_hi)
            sv    = scale_vector.copy()
            sv[k] = s_mid
            gmm_p = build_perturbed_gmm(md.gmm_prop, sv)
            Tb_pl, Tb_ph = sample_tb_percentiles(gmm_p, md,
                                                  n_samples=N_SAMPLES_BISECT,
                                                  desc=f"  dim {k} s={s_mid:.3f}")
            ft, ff = consistency_scores(Tb_pl, Tb_ph, md)

            if ft >= CONSISTENCY_TARGET and ff >= CONSISTENCY_TARGET:
                s_best = s_mid   # still consistent → reduce further
                s_hi   = s_mid
            else:
                s_lo   = s_mid   # inconsistent → need more variance

            if (s_hi - s_lo) < BISECT_TOL:
                break

        reduction = scale_vector[k] - s_best
        scale_vector[k] = s_best
        dim_bar.set_postfix({
            "dim":       k,
            "scale":     f"{s_best:.4f}",
            "reduced_by":f"{reduction:.4f}"
        })
        if verbose:
            print(f"  Dim {k:3d}: {s_global:.4f} → {s_best:.4f}  "
                  f"(reduced by {reduction:.4f})")

    return scale_vector


# ── Main ──────────────────────────────────────────────────────────────────────
def inflate_gmm_covariance(md, refine=True, verbose=True):
    """
    Phase 1 : 1D global bisection  — finds minimal uniform scale s* across
              all PCA dims that achieves CONSISTENCY_TARGET.
    Phase 2 : Per-dim refinement   — starting from s*, tries to reduce each
              dim back toward 1 while keeping consistency (LASSO spirit).
    Phase 3 : Final full evaluation at N=2000.

    Parameters
    ----------
    refine : bool
        If True, run Phase 2 refinement. If False, return uniform s* directly.
        Set False for a quick result; True for a more parsimonious solution.
    """
    n_dims = md.pca_x.n_components_

    # ── Baseline ──────────────────────────────────────────────────────────────
    print("── Phase 0: Baseline ────────────────────────────────────────────────")
    gmm0 = build_perturbed_gmm(md.gmm_prop, np.ones(n_dims))
    Tb_pl0, Tb_ph0 = sample_tb_percentiles(gmm0, md,
                                            n_samples=N_SAMPLES_BISECT,
                                            desc="Baseline")
    ft0, ff0 = consistency_scores(Tb_pl0, Tb_ph0, md)
    print(f"Baseline — Thawed: {ft0:.3f}  Frozen: {ff0:.3f}\n")

    # ── Phase 1 ───────────────────────────────────────────────────────────────
    s_global, ft_g, ff_g = find_global_scale(md, verbose=verbose)

    # ── Phase 2 (optional) ────────────────────────────────────────────────────
    if refine:
        scale_vector = refine_per_dim(md, s_global, verbose=verbose)
    else:
        scale_vector = np.full(n_dims, s_global)
        print(f"\nSkipping refinement — uniform scale s*={s_global:.4f} on all dims.")

    # ── Phase 3: final evaluation ─────────────────────────────────────────────
    print(f"\n── Phase 3: Final evaluation (N={N_SAMPLES_FINAL}) ──────────────────────────")
    gmm_inflated = build_perturbed_gmm(md.gmm_prop, scale_vector)
    Tb_pl_f, Tb_ph_f = sample_tb_percentiles(gmm_inflated, md,
                                              n_samples=N_SAMPLES_FINAL,
                                              desc="Final eval")
    ft_f, ff_f = consistency_scores(Tb_pl_f, Tb_ph_f, md)
    lasso      = np.sum((scale_vector - 1.0) ** 2)
    n_inflated = np.sum(scale_vector > 1.0 + BISECT_TOL)

    print(f"Final — Thawed: {ft_f:.3f}  Frozen: {ff_f:.3f}")
    print(f"LASSO penalty Σ(s−1)²    : {lasso:.4f}")
    print(f"Dims inflated (s > 1.001): {n_inflated} / {n_dims}")

    history = [dict(dim=k, scale=scale_vector[k]) for k in range(n_dims)]
    return gmm_inflated, scale_vector, history


# ── Run — set refine=False for speed, True for parsimony ─────────────────────
gmm_inflated, scale_vector, history = inflate_gmm_covariance(
    md, refine=False, verbose=True   # flip to True once global scale looks good
)


In [ ]:
# ── Compute Tb percentiles from inflated GMM at full resolution ───────────────
Tb_plow_inf, Tb_phigh_inf = sample_tb_percentiles(
    gmm_inflated, md, n_samples=N_SAMPLES_FINAL, desc="Percentile maps"
)

# ── Reconstruct spatial maps ──────────────────────────────────────────────────
domain_idx = np.where(md.domain_mask.flatten())[0]
full_shape = md.domain_mask.flatten().shape[0]
grid_shape = (md.ny, md.nx)
flat_pmp   = md.pmp.flatten()

def to_map(arr_domain, full_shape, domain_idx, grid_shape):
    m = np.full(full_shape, np.nan)
    m[domain_idx] = arr_domain
    return m.reshape(grid_shape)

pmp_map      = flat_pmp.reshape(grid_shape)
Tb_phigh_map = to_map(Tb_phigh_inf, full_shape, domain_idx, grid_shape)
Tb_plow_map  = to_map(Tb_plow_inf,  full_shape, domain_idx, grid_shape)

# pmp - Tb: positive means below melting point (frozen-ish)
#           near zero / negative means at or above melting point (thawed)
delta_high = pmp_map - Tb_phigh_map   # 95th pct: how close the warm tail gets to pmp
delta_low  = pmp_map - Tb_plow_map    # 5th  pct: how far the cold tail is from pmp

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 7))

# Left: pmp - T_b at 95th percentile
# Values near 0 or negative → warm tail reaches/exceeds melting point (thawed-consistent)
# Large positive → even warm tail is well below melting point
im0 = axes[0].imshow(
    delta_high,
    extent=md.extent, origin='upper',
    cmap='Blues',
    vmin = 0, vmax=5
)
axes[0].plot(md.domain_bound_line[0], md.domain_bound_line[1], color='k', lw=1.5)
axes[0].set_title(r"$T_\mathrm{pmp} - T_b^{95\%}$  (K)"
                  "\n(negative = warm tail exceeds melting point)")
axes[0].set_xlabel("x (km)")
axes[0].set_ylabel("y (km)")
# flip y axis
axes[0].invert_yaxis()
# tilt the x-axis labels by 45 degrees
axes[0].tick_params(axis='x', rotation=45)
cb0 = fig.colorbar(im0, ax=axes[0], label="K")


# Overlay thawed evidence locations
thawed_map = md.thawed_mask.reshape(grid_shape).astype(float)
thawed_map[thawed_map == 0] = np.nan
axes[0].imshow(thawed_map,
                extent=md.extent, origin='upper',
                cmap='Reds',
                vmin=0, vmax=1)
axes[0].set_title(r"$T_\mathrm{pmp} - T_b^{95\%}$  (K)"
                  "\nRed = thawed evidence")

axes[0].invert_yaxis()

# Right: pmp - T_b at 5th percentile
# Large positive → cold tail is well below melting point (frozen-consistent)
im1 = axes[1].imshow(
    delta_low,
    extent=md.extent, origin='upper',
    cmap='Blues',
    vmin=0, vmax=20
)
axes[1].set_title(r"$T_\mathrm{pmp} - T_b^{5\%}$  (K)"
                  "\n(large = cold tail well below melting point)")
axes[1].plot(md.domain_bound_line[0], md.domain_bound_line[1], color='k', lw=1.5)
axes[1].set_xlabel("x (km)")
axes[1].set_ylabel("y (km)")
axes[1].tick_params(axis='x', rotation=45)
axes[1].invert_yaxis()

cb1 = fig.colorbar(im1, ax=axes[1], label="K")

# Overlay frozen evidence locations
frozen_map = md.frozen_mask.reshape(grid_shape).astype(float)
frozen_map[frozen_map == 0] = np.nan
axes[1].imshow(frozen_map,
                extent=md.extent, origin='upper',
                cmap='Oranges',
                vmin=0, vmax=1)
axes[1].set_title(r"$T_\mathrm{pmp} - T_b^{5\%}$  (K)"
                  "\nOrange = frozen evidence")

axes[1].invert_yaxis()

plt.suptitle("Inflated Prior Predictive — 90% Interval of $T_b$\n",
             fontsize=12)
plt.tight_layout()
plt.savefig('../figs/inflated_prior_predictive.png', dpi=300)
plt.show()


In [ ]:
# ── Generate enthalpy samples from both GMMs ──────────────────────────────────
N_PLOT_SAMPLES = 500   # enough for stable mean/std maps, fast to compute

def gmm_to_enthalpy_samples(gmm, md, n_samples=N_PLOT_SAMPLES):
    """
    Draw n_samples from gmm, push through PCA inverse + reverse_standardize.
    Returns Eb_ori of shape (n_grid_flat, n_samples) to match existing code convention.
    """
    samples  = gmm.sample(n_samples)                        # (n_samples, n_pca)
    Eb_batch = md.pca_x.inverse_transform(samples)          # (n_samples, n_grid_flat)

    Eb_ori_all = np.zeros_like(Eb_batch)
    for i in tqdm(range(n_samples), desc="Eb samples", leave=False):
        Eb_ori_all[i] = reverse_standardize(Eb_batch[i], md.X_mean, md.X_std,
                                             method='relaxation',
                                             epsilon=md.X_epsilon)

    return Eb_ori_all.T    # → (n_grid_flat, n_samples)  matches existing convention


print("Sampling from original GMM...")
Eb_orig_samples = gmm_to_enthalpy_samples(
    build_perturbed_gmm(md.gmm_prop, np.ones(md.pca_x.n_components_)), md
)

print("Sampling from inflated GMM...")
Eb_infl_samples = gmm_to_enthalpy_samples(gmm_inflated, md)


# ── Compute mean / std maps ───────────────────────────────────────────────────
def make_mean_std_maps(Eb_samples, md):
    """Returns (mean_map, std_map) each shape (nx, ny) with NaN outside domain."""
    mean_map = np.mean(Eb_samples, axis=1).reshape(md.nx, md.ny)
    std_map  = np.std( Eb_samples, axis=1).reshape(md.nx, md.ny)
    mean_map[md.domain_mask == False] = np.nan
    std_map[ md.domain_mask == False] = np.nan
    return mean_map, std_map

mean_orig, std_orig = make_mean_std_maps(Eb_orig_samples, md)
mean_infl, std_infl = make_mean_std_maps(Eb_infl_samples, md)


# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(10, 10))

# shared color limits
vmax_std  = max(np.nanmax(std_orig),  np.nanmax(std_infl))
vmax_mean = max(np.nanmax(np.abs(mean_orig)), np.nanmax(np.abs(mean_infl)))

# ── Row 1: original GMM ───────────────────────────────────────────────────────
im0 = axes[0, 0].imshow(mean_orig, cmap='bwr',
                         vmin=40000, vmax=100000)
axes[0, 0].set_title(r'Mean of $E_b$ (Original GMM)')
axes[0, 0].invert_yaxis()
fig.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(std_orig, cmap='hot',
                         vmin=0, vmax=vmax_std)
axes[0, 1].set_title(r'Std of $E_b$ (Original GMM)')
axes[0, 1].invert_yaxis()
fig.colorbar(im1, ax=axes[0, 1])

# ── Row 2: inflated GMM ───────────────────────────────────────────────────────
im2 = axes[1, 0].imshow(mean_infl, cmap='bwr',
                         vmin=40000, vmax=100000)
axes[1, 0].set_title(r'Mean of $E_b$ (Inflated GMM)')
axes[1, 0].invert_yaxis()
fig.colorbar(im2, ax=axes[1, 0])

im3 = axes[1, 1].imshow(std_infl, cmap='hot',
                         vmin=0, vmax=vmax_std)
axes[1, 1].set_title(r'Std of $E_b$ (Inflated GMM)')
axes[1, 1].invert_yaxis()
fig.colorbar(im3, ax=axes[1, 1])

# ── Row 3: histograms ─────────────────────────────────────────────────────────
domain_m = md.domain_mask.astype(bool)

axes[2, 0].hist(mean_orig[domain_m].flatten(), bins=50,
                color='steelblue', alpha=0.7, label='Original')
axes[2, 0].hist(mean_infl[domain_m].flatten(), bins=50,
                color='tomato',    alpha=0.7, label='Inflated')
axes[2, 0].set_xlabel(r'Mean $E_b$')
axes[2, 0].set_ylabel('Count')
axes[2, 0].set_title(r'Histogram — Mean $E_b$')
axes[2, 0].legend()

axes[2, 1].hist(std_orig[domain_m].flatten(), bins=50,
                color='steelblue', alpha=0.7, label='Original')
axes[2, 1].hist(std_infl[domain_m].flatten(), bins=50,
                color='tomato',    alpha=0.7, label='Inflated')
axes[2, 1].set_xlabel(r'Std $E_b$')
axes[2, 1].set_ylabel('Count')
axes[2, 1].set_title(r'Histogram — Std $E_b$')
axes[2, 1].legend()

plt.suptitle(r"$E_b$ distribution: Original vs Inflated GMM  ($s^* = 10$)",
             fontsize=13)
plt.tight_layout()
plt.show()


# Prior predictive check

This is used to build a randomly thinned basal evidence data. Since we are adding low rank GP, this is probably not needed anymore in the standard workflow.

In [ ]:
# ── Basal Evidence Consistency Check ─────────────────────────────────────────
# Thawed: 99th percentile of Tb must be below pmp - err_eps  (i.e. Tb_high < pmp - ε)
# Frozen:  1st percentile of Tb must be above pmp + err_eps  (i.e. Tb_low  > pmp + ε)

err_eps = 1

flat_shape  = md.thawed_mask.size  
thawed_mask = md.thawed_mask.flatten().astype(bool)
frozen_mask = md.frozen_mask.flatten().astype(bool)

# ── Percentile deltas relative to pressure-melting point ─────────────────────
thawed_delta = np.full(flat_shape, np.nan)
frozen_delta = np.full(flat_shape, np.nan)

thawed_delta[thawed_mask] = Tb_phigh_map.flatten()[thawed_mask] - md.pmp.flatten()[thawed_mask]
frozen_delta[frozen_mask] = Tb_plow_map.flatten()[frozen_mask]  - md.pmp.flatten()[frozen_mask]

# ── Consistency flags ─────────────────────────────────────────────────────────
thawed_consistent = np.where(thawed_mask, thawed_delta > -err_eps, np.nan)
frozen_consistent = np.where(frozen_mask,  frozen_delta  < -err_eps, np.nan)

# ── Plot ──────────────────────────────────────────────────────────────────────
grid_shape = (md.ny, md.nx)
vlims = [(0, 3), (0, 10)]
plot_pairs = [
    (md.pmp.reshape(md.ny, md.nx) - Tb_phigh_map.reshape(md.ny, md.nx), thawed_consistent, "Thawed Prior Consistency", vlims[0]),
    (md.pmp.reshape(md.ny, md.nx) - Tb_plow_map.reshape(md.ny, md.nx), frozen_consistent,  "Frozen Prior Consistency", vlims[1]),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (bg, data, title, vlim) in zip(axes, plot_pairs):
    imbg = ax.imshow(bg.reshape(grid_shape), extent=md.extent,
                     origin='upper', cmap='hot',
                     vmin=vlim[0], vmax=vlim[1],
                     alpha=0.5, zorder=1)
    im = ax.imshow(data.reshape(grid_shape), extent=md.extent,
                   origin='upper', cmap='coolwarm_r',
                   vmin=0, vmax=1,
                   alpha=1, zorder=2)
    ax.plot(md.domain_bound_line[0], md.domain_bound_line[1], color='k', lw=1.5, zorder=3)
    # invert y axis
    ax.invert_yaxis()
    ax.set_title(title)
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# subselect the valid thawed and frozen evidence based on this prior check
new_thawed_mask = np.where(thawed_consistent == 1, True, False)
new_frozen_mask = np.where(frozen_consistent == 1, True, False)
# convert to 0 and 1
new_thawed_mask = new_thawed_mask.astype(int)
new_frozen_mask = new_frozen_mask.astype(int)
# make new plots for visual inspection
plt.figure(figsize=(12, 5))
plt.subplot(1,2,1)
plt.imshow(new_thawed_mask.reshape(grid_shape), extent=md.extent, origin='upper', cmap='Greys',
           vmin=0, vmax=1)
plt.plot(md.domain_bound_line[0], md.domain_bound_line[1], color='k', lw=1.5)
plt.gca().invert_yaxis()
plt.title("New Thawed Mask")
plt.colorbar()

plt.subplot(1,2,2)
plt.imshow(new_frozen_mask.reshape(grid_shape), extent=md.extent, origin='upper', cmap='Greys',
           vmin=0, vmax=1)
plt.plot(md.domain_bound_line[0], md.domain_bound_line[1], color='k', lw=1.5)
plt.gca().invert_yaxis()
plt.title("New Frozen Mask")
plt.colorbar()

plt.tight_layout()
plt.show()

# do the same for fractional area
new_thawed_fractional_area = np.where(thawed_consistent == 1, thawed_fractional_area, 0)
new_frozen_fractional_area = np.where(frozen_consistent == 1, frozen_fractional_area, 0)



# export the new masks and fractional areas to .npz for later use
np.savez("../data/final-obs-evidence/thawed_frozen_masks.npz",
         new_thawed_mask=new_thawed_mask,
         new_frozen_mask=new_frozen_mask,
         new_thawed_fractional_area=new_thawed_fractional_area,
         new_frozen_fractional_area=new_frozen_fractional_area,
         x = md.coord[0],
         y = md.coord[1])
